# Reconhecimento de Comandos de Voz na Borda

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/kws_borda_relatorio.ipynb)

**Projeto final — IA025 Deep Learning (2026 S2)**
**Equipe:** Guilherme Pulizzi · Jefferson Abraham · Luciano Pinheiro

**Repositório:** [gpeuzinCosta/kws-borda](https://github.com/gpeuzinCosta/kws-borda)

---

Este notebook é **o projeto inteiro em um só lugar**. Lido de cima a baixo, ele explica o problema, as
hipóteses, os dados, cada passo da metodologia, o que já foi feito e o que ainda falta. Ele também reúne
**todos os resultados finais**. A lógica pesada fica nas pastas do repositório (`dados/`, `modelos/`,
`treino/`, `avaliacao/`), e aqui ficam só as chamadas e a análise.

Para rodar: *Ambiente de execução → GPU* e depois *Executar tudo*. Cada seção que depende de algo ainda
não implementado é pulada com um aviso, então o notebook roda inteiro em qualquer fase do projeto.

**Status:** cada seção indica entre parênteses se está (feito), (em andamento) ou (a fazer).

## Sumário

- [Como o repositório está organizado](#organizacao)
- [0. Setup](#setup)
- [1. Problema, hipóteses e métricas](#problema)
- [2. Dados — Google Speech Commands v2](#dados)
- [3. Pré-processamento: MFCC](#mfcc)
- [4. Modelos](#modelos)
- [5. Treino](#treino)
- [6. Resultados no GSC — hipótese (a)](#resultados)
- [7. Matriz de confusão](#confusao)
- [8. Latência: MFCC × rede — hipótese (b)](#latencia)
- [9. Robustez a ruído real (DEMAND)](#ruido)
- [10. Generalização: AudioMNIST](#audiomnist)
- [11. Transferência de idioma: MSWC-pt](#mswc)
- [12. Conclusões](#conclusoes)
- [13. Limitações e próximos passos](#limitacoes)
- [Referências](#referencias)

<a name="organizacao"></a>
## Como o repositório está organizado

```
kws-borda/
├── notebooks/     ← este notebook (projeto + resultados finais) e experimentos/ (rascunhos)
├── modelos/       ← uma arquitetura por arquivo: mlp, cnn1d, resnet, dscnn, atencao
├── dados/         ← funções para gerir os datasets: download, MFCC, GSC v2, DEMAND, AudioMNIST, MSWC-pt
├── treino/        ← loop de treino ÚNICO (loop.py) + script treinar.py
├── avaliacao/     ← métricas, custo (params/MACs), latência, ruído/generalização, gráficos + script avaliar.py
└── configs/       ← base.yaml (regras COMUNS a todos) + <modelo>.yaml (só o que muda)
```

**Por que assim?** Para a comparação ser justa, todas as redes recebem **exatamente** os mesmos dados, o
mesmo MFCC, o mesmo loop de treino e as mesmas métricas. A única coisa que muda é a arquitetura. Cada
integrante implementa o seu modelo num arquivo próprio, com o contrato:

```python
@registrar_modelo("dscnn")
class DSCNN(torch.nn.Module):
    def __init__(self, n_classes, n_mfcc=40, n_frames=101, **hiperparametros): ...
    def forward(self, x):        # x: (B, n_mfcc, n_frames) = MFCC normalizado (40×101 ou 10×49)
        return logits            # (B, n_classes)
```

Feito isso, o modelo aparece sozinho em todas as seções deste notebook.

<a name="setup"></a>
## 0. Setup (feito)

A célula abaixo prepara o ambiente. No Colab, ela monta o Google Drive, clona o repositório e instala
as dependências. Os dados brutos ficam no disco local da sessão (leitura rápida), e o cache de MFCC, os
checkpoints e os resultados ficam no Drive (sobrevivem quando o runtime cai).

In [ ]:
#@title Setup — rode esta célula primeiro
#Funciona no Colab (clona o repo e usa o Drive) e localmente/VS Code (usa a pasta do repositório).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/gpeuzinCosta/kws-borda.git"
NO_COLAB = "google.colab" in sys.modules

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO = Path("/content/kws-borda")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)     #pega a versão mais nova
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

    #Drive: cache de MFCC, runs e resultados PERSISTEM entre sessões
    #Disco local (/content/data): áudios extraídos, leitura rápida (o Drive é lento para 100 mil arquivos)
    os.environ["KWS_DRIVE"] = "/content/drive/MyDrive/kws-borda"
    os.environ["KWS_DATA"] = "/content/data"
else:
    #Local: sobe a partir da pasta atual até achar a raiz do repositório
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

#A raiz do repo no sys.path deixa importar as pastas: configs, dados, modelos, treino, avaliacao
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
from configs import pastas_projeto, escolher_device

PASTAS = pastas_projeto()
DOWNLOADS = Path(os.environ.get("KWS_DRIVE", REPO)) / "downloads"     #.tar.gz/.zip baixados UMA vez
device = escolher_device()
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

print(f"Colab: {NO_COLAB} | device: {device} | commit do código: {commit}")
print(f"torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—'}")
for nome, p in PASTAS.items():
    print(f"  {nome:10s} -> {p}")

### Painel de controle do experimento

Os únicos parâmetros que você precisa mexer para rodar o notebook. As três primeiras escolhas (tarefa,
MFCC e normalização) estão explicadas nas §2 e §3. Cada combinação tem os próprios caches, treinos e
linhas no `resultados.csv`, então dá para rodar todas e comparar. Para um teste rápido de ponta a ponta,
use `MODELOS = ["mlp"]`, `SEEDS = [0]` e `EPOCAS = 1`.

In [ ]:
from configs import PRESETS_MFCC, carregar_config, rotulo_experimento

#Tarefa: "gsc35" (35 palavras; necessária para §10 e §11) ou "gsc12" (10 comandos + unknown + silence,
#comparável aos artigos)
TAREFA = "gsc35"

#MFCC: "padrao" (40 coef × 101 janelas) ou "hello_edge" (10 × 49, igual ao artigo e ao notebook do Luciano)
MFCC = "padrao"

#Normalização: "global" (estatísticas do treino) ou "por_amostra" (cada áudio pela sua própria média/desvio)
NORMALIZACAO = "global"

#Quais modelos rodar (os não implementados são pulados com aviso)
MODELOS = ["mlp", "cnn1d", "resnet", "dscnn", "atencao"]

#Rodadas independentes (sementes diferentes) para média ± desvio
SEEDS = [0, 1, 2]

#None = usa o valor do configs/base.yaml; um número sobrescreve para todos os modelos
EPOCAS = None

#False: se já existir um treino salvo no Drive para (modelo, seed), carrega em vez de treinar de novo
RETREINAR = False

#Seções que dependem de datasets extras (cada uma baixa o seu na primeira vez)
RODAR_RUIDO = True        #§9  DEMAND (~1 GB, 8 ambientes)
RODAR_AUDIOMNIST = True   #§10 AudioMNIST (~1 GB)
RODAR_MSWC = True         #§11 MSWC-pt (1,1 GB)


def config(nome, **extra):
    """Hiperparâmetros do modelo `nome` com as escolhas do painel aplicadas."""
    return carregar_config(nome, tarefa=TAREFA, normalizacao=NORMALIZACAO, **PRESETS_MFCC[MFCC], **extra)


print("experimento:", rotulo_experimento(config("mlp")))

<a name="problema"></a>
## 1. Problema, hipóteses e métricas (feito)

### O problema
As pessoas querem que seus aparelhos respondam **instantaneamente** à voz, em qualquer lugar e de forma
confiável, mesmo sem internet. Isso pede reconhecer comandos curtos (*keyword spotting*: "yes", "stop",
"up"…) **no próprio dispositivo** (na *borda*). Esses dispositivos têm pouca memória (KB, não GB), CPUs
fracas e bateria limitada.

### O que vamos investigar
Vamos pegar as famílias de redes implementadas na disciplina (**MLP, CNN 1D, ResNet, DS-CNN e Atenção**),
encolher cada uma para o tamanho de um dispositivo restrito e colocá-las na **mesma bancada de testes**.

**Hipótese (a):** é possível encolher essas arquiteturas clássicas e ainda assim manter um reconhecimento
de voz funcional?
→ respondida na §6 (acurácia × tamanho/MACs).

**Hipótese (b):** em modelos tão pequenos, o que consome mais recursos, o **pré-processamento** do áudio
(MFCC) ou a **inferência** da rede?
→ respondida na §8 (latência medida separadamente).

### Métricas

| Métrica | O que mede | Por que importa aqui |
|---|---|---|
| **Acurácia** | fração de comandos reconhecidos corretamente | desempenho geral |
| **F1 macro** | média do F1 de cada classe (F1 = média harmônica de precisão e revocação) | dá o mesmo peso a todas as 35 palavras, e as difíceis não somem atrás das fáceis |
| **Matriz de confusão** | quais palavras viram quais | mapeia os erros **fonéticos** (tree ↔ three) |
| **Parâmetros / KB** | nº de pesos; tamanho em fp32 (×4 bytes) e int8 (×1 byte) | cabe na memória do dispositivo? |
| **MACs** | multiplicações + somas para classificar **um** áudio | custo matemático, independente do hardware |
| **Latência (ms)** | tempo real na CPU, batch = 1, 1 thread, **MFCC e rede separados** | responde (b) diretamente |

<a name="dados"></a>
## 2. Dados — Google Speech Commands v2 (feito)

O **GSC v2** (Warden, 2018) é o dataset padrão de *keyword spotting*:
- **105.835** áudios de **1 segundo**, gravados a **16 kHz** por ~2.600 pessoas, cada um com uma palavra;
- **35 palavras**: comandos (*yes, no, up, down, left, right, on, off, stop, go, forward, backward, follow, learn*),
  dígitos (*zero … nine*) e palavras auxiliares (*bed, bird, cat, dog, happy, house, marvin, sheila, tree, visual, wow*);
- **split oficial por locutor** (`validation_list.txt`, `testing_list.txt`): quem fala no teste nunca aparece
  no treino, então medimos a generalização para **vozes novas**.

### Duas tarefas (`TAREFA` no painel)

| Tarefa | Classes | Para que serve |
|---|---|---|
| `gsc35` | as 35 palavras | necessária para o AudioMNIST (§10, dígitos) e para o mapeamento português → inglês (§11) |
| `gsc12` | 10 comandos (*yes, no, up, down, left, right, on, off, stop, go*) + `_unknown_` + `_silence_` | o recorte **padrão da literatura** (Hello Edge, KWT): nossos números ficam comparáveis aos artigos |

A montagem da `gsc12` veio do notebook do Luciano (`experimentos/Projeto_MLP.ipynb`):
- **`_unknown_`**: sorteio das outras 25 palavras, na mesma quantidade média de cada comando. Ensina o modelo a dizer "não é um comando" em vez de forçar uma das 10 palavras.
- **`_silence_`**: trechos de 1 s dos ruídos de fundo que vêm com o GSC (`_background_noise_`), com volume reduzido a 10%. Ensina o modelo a não disparar quando ninguém fala.

O sorteio usa semente fixa: todos os integrantes e todos os modelos veem exatamente os mesmos arquivos.

A célula abaixo baixa o `.tar.gz` (2,3 GB) para o Drive **uma vez** e o extrai no disco local a cada sessão
(~1–2 min).

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "gsc",
                "--destino", str(PASTAS["dados"]), "--arquivos", str(DOWNLOADS)], check=True)
RAIZ_GSC = PASTAS["dados"] / "gsc"

### Um exemplo do dataset

Cada áudio é um vetor de 16.000 números (1 s × 16 kHz) na escala PCM [−1, 1]. Esses números são a
representação digital da tensão captada pelo microfone.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio
from dados import classes_da_tarefa, listar_split, ler_trecho, rotulo

CLASSES = classes_da_tarefa(TAREFA)
split = listar_split(RAIZ_GSC, TAREFA)
print({k: len(v) for k, v in split.items()})

exemplo = split["treino"][0]
wav = ler_trecho(RAIZ_GSC, exemplo)
print("arquivo:", exemplo, "| formato:", tuple(wav.shape), "| palavra:", exemplo.split("/")[0])

plt.figure(figsize=(10, 2.5))
plt.plot(wav.numpy()); plt.title(f"Forma de onda — '{exemplo.split('/')[0]}'")
plt.xlabel("amostra (16 kHz)"); plt.ylabel("amplitude"); plt.grid(alpha=0.3); plt.show()
Audio(wav.numpy(), rate=16000)

In [ ]:
#Distribuição das classes no treino. Na gsc35 o GSC NÃO é balanceado (as palavras auxiliares têm menos
#exemplos); na gsc12 unknown e silence são montados com a quantidade média de um comando
import collections
contagem = collections.Counter(CLASSES[rotulo(rel, TAREFA)] for rel in split["treino"])

plt.figure(figsize=(14, 3.5))
plt.bar(CLASSES, [contagem[c] for c in CLASSES])
plt.xticks(rotation=90); plt.ylabel("nº de áudios"); plt.title("Áudios de treino por palavra")
plt.grid(axis="y", alpha=0.3); plt.show()

**O que observamos:** _(preencher: classes menos frequentes, implicação para usar o F1 macro…)_

<a name="mfcc"></a>
## 3. Pré-processamento: MFCC (feito)

A forma de onda crua tem 16.000 números por segundo, e a maior parte é irrelevante para saber **qual
palavra** foi dita (fase, ruído de alta frequência…). O **MFCC** (*Mel-Frequency Cepstral Coefficients*)
resume o áudio assim:

1. corta o áudio em janelas de **30 ms**, com passo de **10 ms** → **101 janelas**;
2. calcula o espectro (FFT) de cada janela;
3. agrupa as frequências em **40 filtros na escala Mel**, que imita a audição humana: mais resolução nos graves;
4. aplica log e uma DCT → **40 coeficientes** por janela.

Resultado: cada áudio vira uma "imagem" **40 × 101**, que é a entrada de **todas** as redes.

### Duas configurações de MFCC (`MFCC` no painel)

| Preset | Coeficientes | Janela / passo | Entrada | Origem |
|---|---|---|---|---|
| `padrao` | 40 | 30 ms / 10 ms | **40 × 101** (4.040 números) | configuração usual em KWS |
| `hello_edge` | 10 | 40 ms / 20 ms, sem *padding* nas bordas | **10 × 49** (490 números) | artigo Hello Edge e notebook do Luciano |

A `hello_edge` dá uma entrada **8× menor** e um MFCC mais barato. Isso entra direto na hipótese (b): se o
pré-processamento domina o custo, encolher o MFCC pode valer mais que encolher a rede.

### Duas normalizações (`NORMALIZACAO` no painel)

- **`global`**: média e desvio **por coeficiente, calculados só no treino** (medir na validação/teste seria vazamento). O dispositivo precisa guardar esses 2 × n_mfcc números.
- **`por_amostra`** (notebook do Luciano): cada áudio é normalizado pela **própria** média e desvio ao longo do tempo. O dispositivo não guarda nada, e o modelo fica menos sensível ao volume do microfone.

Esse passo tem um custo, e medir esse custo é justamente a **hipótese (b)**. Por isso o MFCC fica isolado
em `dados/mfcc.py` (`ExtratorMFCC`), um `nn.Module` que pode ser cronometrado como uma rede.

**Decisões:**
- a biblioteca é o **torchaudio**; como a implementação do MFCC pode mudar a resposta de (b), a §8 mede também o librosa;
- o MFCC de todo o dataset é calculado **uma vez** e salvo em cache no Drive;
- o **SpecAugment** leve no treino zera uma faixa de tempo e uma de coeficientes. A mesma regularização vale para todos os modelos.

In [ ]:
from dados import ExtratorMFCC

hp_base = config("mlp")      #as seções dados/treino são iguais para todos os modelos
extrator = ExtratorMFCC.de_hparams(hp_base)

mfcc = extrator(wav)[0]
print("MFCC:", tuple(mfcc.shape), "(coeficientes × janelas)")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(wav.numpy()); axes[0].set_title("Forma de onda (16.000 amostras)")
im = axes[1].imshow(mfcc.numpy(), aspect="auto", origin="lower", cmap="magma")
axes[1].set_title(f"MFCC ({mfcc.shape[0]} × {mfcc.shape[1]})"); axes[1].set_xlabel(f"janela ({hp_base['passo_ms']} ms)"); axes[1].set_ylabel("coeficiente")
fig.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

In [ ]:
%%time
#Calcula (ou carrega do cache no Drive) o MFCC de treino/val/teste — primeira vez: ~10 min
from dados import carregar_gsc, criar_dataloaders

dados = carregar_gsc(hp_base, RAIZ_GSC, PASTAS["cache"])
for k, (X, y) in dados.items():
    print(f"{k:6s} | X {tuple(X.shape)} | y {tuple(y.shape)}")

In [ ]:
#Confere o batch que vai entrar na rede: formato e normalização (média ~0, desvio ~1)
#Com normalização por amostra, MEDIA e DESVIO são None (nada a guardar)
loaders, (MEDIA, DESVIO) = criar_dataloaders(hp_base, dados)
xb, yb = next(iter(loaders["treino"]))
print("batch:", tuple(xb.shape), "| média:", round(xb.mean().item(), 3), "| desvio:", round(xb.std().item(), 3))
print("palavras:", [CLASSES[i] for i in yb[:8].tolist()])

<a name="modelos"></a>
## 4. Modelos (em andamento)

As 5 famílias da disciplina, em versões **pequenas**. A tabela abaixo é gerada a partir do registro de
modelos (`modelos/__init__.py`), então um modelo novo aparece aqui sozinho.

In [ ]:
import pandas as pd
from modelos import listar_modelos

pd.DataFrame(listar_modelos())

### 4.1 MLP (feito)
**Responsável:** Guilherme · **Código:** `modelos/mlp.py` · **Config:** `configs/mlp.yaml`

Achata o MFCC (40 × 101 = 4.040 números) e passa por camadas densas. Não tem noção de tempo nem de
vizinhança: cada coeficiente em cada janela tem um peso próprio. **Papel no projeto:** é o *baseline*. Esperamos
que tenha **muitos parâmetros e acurácia baixa**, um contraexemplo útil para (a): tamanho não é tudo, a
arquitetura importa.

### 4.2 CNN 1D (a fazer)
**Responsável:** _a definir_ · **Código:** `modelos/cnn1d.py` · **Config:** `configs/cnn1d.yaml`

Os 40 coeficientes viram **canais**, e a convolução desliza só no **tempo**. Cada filtro aprende um padrão
espectral que pode aparecer em qualquer momento da palavra (compartilhamento de pesos). Muito barata em
MACs.

### 4.3 ResNet (a fazer)
**Responsável:** _a definir_ · **Código:** `modelos/resnet.py` · **Config:** `configs/resnet.yaml`

Trata o MFCC como uma imagem 1 × 40 × 101 (Conv2d). Os **atalhos residuais** (`x + F(x)`) deixam o
gradiente fluir e permitem empilhar mais camadas com poucos canais (estilo *res8*, Tang & Lin 2018).

### 4.4 DS-CNN (a fazer)
**Responsável:** _a definir_ · **Código:** `modelos/dscnn.py` · **Config:** `configs/dscnn.yaml`

Convolução **separável em profundidade** (Hello Edge, Zhang et al. 2017). Uma conv 3×3 comum com C canais
custa C·C·9 pesos. A separável divide em *depthwise* (um filtro 3×3 por canal: C·9) + *pointwise* (conv 1×1
que mistura canais: C·C), com ~8–9× menos parâmetros. Foi feita para **microcontroladores**.

### 4.5 Atenção (KWT) (a fazer)
**Responsável:** _a definir_ · **Código:** `modelos/atencao.py` · **Config:** `configs/atencao.yaml`

*Keyword Transformer* (Berg et al. 2021). Cada janela do MFCC vira um **token**, como uma palavra numa
frase ou um patch no Vision Transformer. A **autoatenção** deixa cada janela "olhar" para todas as outras,
e um token `[CLS]` aprendido resume o áudio.
Cuidado: em `eval()` o PyTorch usa um kernel fundido que o contador de MACs não vê, então contamos em `train()`.

### Custo computacional de cada modelo

Parâmetros, tamanho e MACs dependem **só da arquitetura**, então dá para medir antes de treinar.

In [ ]:
from avaliacao import custo
from modelos import criar_modelo, esta_implementado

linhas = []
for nome in MODELOS:
    if not esta_implementado(nome):
        print(f"[pulando] {nome}: ainda não implementado")
        continue
    hp = config(nome)
    linhas.append(dict(modelo=nome, **custo(criar_modelo(nome, hp))))

tabela_custo = pd.DataFrame(linhas)
tabela_custo.style.format({"params": "{:,}", "kb_fp32": "{:.0f}", "kb_int8": "{:.0f}", "macs": "{:,}"})

<a name="treino"></a>
## 5. Treino (em andamento)

**O mesmo loop para todas as redes** (`treino/loop.py`):

| Item | Valor (configs/base.yaml) | Por quê |
|---|---|---|
| Otimizador | AdamW, lr 3e-3, weight decay 1e-2 | robusto e com regularização L2 desacoplada |
| Agendador | OneCycle (por batch) | aquece o lr no início e desce suavemente: converge rápido |
| Perda | entropia cruzada com *label smoothing* 0,1 | evita confiança excessiva |
| Aumento de dados | SpecAugment leve | regularização barata |
| Seleção | pesos da **melhor época na validação** | *early stopping* passivo |
| Repetições | 3 seeds por modelo | média ± desvio (validação estatística) |

Cada modelo pode ajustar seus próprios hiperparâmetros no `configs/<modelo>.yaml` (ex.: lr menor para a
Atenção). Cada treino fica salvo no Drive (`runs/<modelo>_s<seed>_<data>/`). Com `RETREINAR = False`, rodar
o notebook de novo **carrega** os treinos em vez de refazê-los.

In [ ]:
from treino import treinar, carregar_run
from avaliacao.graficos import curvas_treino

treinados = {}   #{nome: [(seed, model, hist), ...]}

for nome in MODELOS:
    if not esta_implementado(nome):
        print(f"[pulando] {nome}: ainda não implementado")
        continue
    hp = config(nome, epocas=EPOCAS)
    loaders_m, _ = criar_dataloaders(hp, dados)      #batch_size pode mudar por modelo
    treinados[nome] = []
    for seed in SEEDS:
        print(f"\n===== {nome} | seed {seed} =====")
        run = None if RETREINAR else carregar_run(PASTAS["runs"], hp, seed)
        model = criar_modelo(nome, hp)
        if run is not None:
            model.load_state_dict(run[0]); hist = run[1]
            print(f"carregado do Drive: {hist.get('pasta')} | melhor acc val {max(hist['acc'])*100:.2f}%")
        else:
            model, hist = treinar(model, loaders_m, hp, seed=seed, pasta_runs=PASTAS["runs"], device=device)
        treinados[nome].append((seed, model.to(device), hist))

In [ ]:
#Curvas de treino da primeira seed de cada modelo
for nome, rodadas in treinados.items():
    curvas_treino(rodadas[0][2], titulo=f"({nome}, seed {rodadas[0][0]})")

**O que observamos:** _(preencher: convergência, overfitting — gap entre treino e val —, nº de épocas suficiente…)_

<a name="resultados"></a>
## 6. Resultados no GSC — hipótese (a) (a fazer)

Avaliação no conjunto de **teste** (11.005 áudios de locutores nunca vistos), com média ± desvio entre as
seeds. Cada rodada também é registrada em `runs/resultados.csv` no Drive, a tabela de experimentos do grupo.

**Critério para "funcional":** acurácia mínima de _(definir: ex. 90% nas 35 classes)_.

In [ ]:
from treino import prever
from avaliacao import acuracia, f1_macro, registrar
from avaliacao.resultados import consolidar

linhas, predicoes = [], {}
for nome, rodadas in treinados.items():
    hp = config(nome)
    loaders_m, _ = criar_dataloaders(hp, dados)
    for seed, model, hist in rodadas:
        y_pred, y_true = prever(model, loaders_m["teste"])
        predicoes[(nome, seed)] = (y_pred, y_true)
        r = dict(acc=acuracia(y_pred, y_true), f1=f1_macro(y_pred, y_true))
        registrar(PASTAS["resultados"] / "resultados.csv", nome, seed, experimento=rotulo_experimento(hp),
                  commit=commit, **r)
        linhas.append(dict(modelo=nome, seed=seed, dataset="gsc_v2", **r))

resultados_gsc = consolidar(pd.DataFrame(linhas))
if len(resultados_gsc):
    resultados_gsc = resultados_gsc.merge(tabela_custo, on="modelo")
resultados_gsc

In [ ]:
from avaliacao.graficos import acuracia_vs_custo

if len(resultados_gsc):
    acuracia_vs_custo(resultados_gsc, x="macs")
    acuracia_vs_custo(resultados_gsc, x="kb_int8")

**O que observamos / resposta à hipótese (a):** _(preencher: quais modelos passam do critério, menor modelo
funcional, quem domina a fronteira acurácia × custo)_

<a name="confusao"></a>
## 7. Matriz de confusão (a fazer)

Linha = palavra verdadeira; coluna = palavra prevista; normalizada por linha. Fora da diagonal estão os
**erros**, e esperamos que sejam **fonéticos**: palavras de som parecido, como *tree* e *three*, *go* e *no*.

Abaixo da matriz vem o **relatório por classe** (trazido do notebook do Luciano), com precisão, revocação
e F1 de cada palavra:
- **precisão**: das vezes que o modelo disse "go", quantas eram "go" de fato?
- **revocação**: dos "go" verdadeiros, quantos o modelo encontrou?

No experimento do Luciano (MLP, 12 classes), `_unknown_` teve revocação de só 0,49 e `go` teve F1 de 0,67.

In [ ]:
from avaliacao import matriz_confusao, relatorio_por_classe, top_confusoes
from avaliacao.graficos import plot_matriz_confusao

if len(resultados_gsc):
    melhor = resultados_gsc.sort_values("acc_mean", ascending=False).iloc[0]["modelo"]
    y_pred, y_true = predicoes[(melhor, SEEDS[0])]
    cm = matriz_confusao(y_pred, y_true, n_classes=len(CLASSES))
    plot_matriz_confusao(cm, CLASSES, titulo=f"Matriz de confusão — {melhor} (seed {SEEDS[0]})")
    display(pd.DataFrame(top_confusoes(cm, CLASSES, k=10), columns=["verdadeira", "prevista", "fração"]))

    #Precisão, revocação e F1 de cada palavra, da pior para a melhor
    display(relatorio_por_classe(y_pred, y_true, CLASSES).round(3))

**O que observamos:** _(preencher: pares fonéticos mais confundidos, se são os mesmos em todos os modelos,
palavras mais fáceis/difíceis)_

<a name="latencia"></a>
## 8. Latência: MFCC × rede — hipótese (b) (a fazer)

**Protocolo** (para ser justo e reprodutível):
- **batch = 1**: um comando por vez, como no dispositivo;
- **CPU com 1 thread**: simula um núcleo de dispositivo de borda (no Colab a GPU é ignorada aqui);
- **aquecimento** + **mediana e p95** de 300 repetições;
- o **MFCC é medido separado** da rede, e com **duas bibliotecas** (torchaudio e librosa).

Medimos o MFCC com duas bibliotecas porque a implementação pode mudar a resposta de (b): numa biblioteca
mais lenta, o pré-processamento pode custar mais que a rede mesmo nos modelos maiores.

In [ ]:
from avaliacao import medir_latencia
from avaliacao.graficos import barras_latencia

linhas = []
for nome in MODELOS:
    if not esta_implementado(nome):
        continue
    hp = config(nome)
    model = treinados[nome][0][1] if nome in treinados else criar_modelo(nome, hp)
    linhas.append(dict(modelo=nome, **medir_latencia(model, ExtratorMFCC.de_hparams(hp),
                                                     threads=hp["latencia_threads"], com_librosa=True)))

latencia = pd.DataFrame(linhas)
if len(latencia):
    display(latencia.round(3))
    barras_latencia(latencia)

**O que observamos / resposta à hipótese (b):** _(preencher: para quais modelos o MFCC domina, efeito da
biblioteca, implicação para a borda)_

<a name="ruido"></a>
## 9. Robustez a ruído real (DEMAND) (a fazer)

**Atenção:** o link do Zenodo do plano (1227121) **não é um dataset de fala**. É o **DEMAND**:
gravações de **ruído ambiente** (cozinha, sala, escritório, cafeteria, trânsito, metrô, carro, parque).

**Como usamos:** misturamos esse ruído aos áudios de **teste** do GSC em vários níveis de **SNR**
(*signal-to-noise ratio*, a relação sinal-ruído):

$$\text{SNR (dB)} = 10 \log_{10} \frac{P_\text{fala}}{P_\text{ruído}}$$

20 dB = ruído quase imperceptível · 0 dB = mesma potência · −5 dB = ruído mais forte que a fala.
Nenhum modelo vê ruído no treino, então a curva mostra quanto cada arquitetura **resiste** fora da
distribuição.

In [ ]:
import random
from dados.ruido import carregar_demand
from avaliacao import avaliar_ruido
from avaliacao.graficos import curvas_ruido

if RODAR_RUIDO and treinados:
    subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "demand",
                    "--destino", str(PASTAS["dados"]), "--arquivos", str(DOWNLOADS)], check=True)
    ruidos = carregar_demand(PASTAS["dados"] / "demand")
    print("ambientes:", list(ruidos))

    #3.000 áudios de teste (subamostra para caber no tempo; aumentar na versão final)
    amostra = random.Random(0).sample(split["teste"], 3000)
    wavs = [ler_trecho(RAIZ_GSC, rel) for rel in amostra]
    y = torch.tensor([rotulo(rel, TAREFA) for rel in amostra])

    snrs = hp_base["snrs_db"]
    modelos_seed0 = {nome: rodadas[0][1] for nome, rodadas in treinados.items()}
    res_ruido = avaliar_ruido(modelos_seed0, wavs, y, ruidos, snrs, MEDIA, DESVIO, extrator)
    curvas_ruido(res_ruido, snrs)
else:
    print("[pulando] §9: RODAR_RUIDO=False ou nenhum modelo treinado")

**O que observamos:** _(preencher: ordem de robustez, em que SNR cada modelo deixa de ser "funcional")_

**Próximo passo possível:** treinar com ruído (*data augmentation*) e comparar.

<a name="audiomnist"></a>
## 10. Generalização: AudioMNIST (a fazer)

**AudioMNIST**: 30.000 gravações dos dígitos 0–9 por **60 falantes**, a 48 kHz. O
microfone é o mesmo para todos; o que varia é a **sala** de gravação e o **sotaque** (57 de 60 não são
nativos de inglês). O teste então mede: *o modelo treinado no GSC reconhece outros falantes, sotaques e salas?*

Os 10 dígitos existem no GSC (*zero … nine*), então avaliamos **sem re-treino** de duas formas:
1. **35 classes**: o modelo pode errar para qualquer palavra;
2. **restrito aos dígitos**: considera só os 10 logits dos dígitos.

Como as gravações são limpas e o vocabulário (10 dígitos) é mais fácil, o acerto pode ficar **maior** que no
próprio GSC. Nesse caso o resultado indica generalização de falante, e não serve como teste de estresse.

In [ ]:
from dados.audiomnist import listar_audiomnist, DIGITOS_EN
from avaliacao import logits_de_wavs

from dados import CLASSES_GSC, ler_audio

if TAREFA != "gsc35":
    print("[pulando] §10: o AudioMNIST precisa da tarefa gsc35 (os dígitos não estão na gsc12)")
elif RODAR_AUDIOMNIST and treinados:
    subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "audiomnist",
                    "--destino", str(PASTAS["dados"])], check=True)
    arquivos, rotulos = listar_audiomnist(PASTAS["dados"] / "audiomnist", por_falante=100)
    wavs_am = [ler_audio(f) for f in arquivos]
    y_am = torch.tensor(rotulos)
    idx_digitos = torch.tensor([CLASSES_GSC.index(d) for d in DIGITOS_EN])
    print(len(wavs_am), "áudios")

    linhas = []
    for nome, rodadas in treinados.items():
        L = logits_de_wavs(rodadas[0][1], wavs_am, extrator, MEDIA, DESVIO)
        linhas.append(dict(modelo=nome,
                           acc_35_classes=(L.argmax(1) == idx_digitos[y_am]).float().mean().item(),
                           acc_so_digitos=(L[:, idx_digitos].argmax(1) == y_am).float().mean().item()))
    display(pd.DataFrame(linhas).round(4))
else:
    print("[pulando] §10: RODAR_AUDIOMNIST=False ou nenhum modelo treinado")

**O que observamos:** _(preencher)_

<a name="mswc"></a>
## 11. Transferência de idioma: MSWC-pt (a fazer)

**Multilingual Spoken Words Corpus**, parte em português. Dígitos, *sim, não, cima,
baixo* e *vai* têm clipes suficientes; *esquerda, direita, parar, ligar, desligar* têm poucos; *um* não
existe (o MSWC exclui palavras com menos de 3 letras).

Dois experimentos:
1. **Zero-shot mapeado**: o modelo em inglês acerta o equivalente (*sim → yes*, *cima → up*)? Esperado: perto do acaso.
2. **Cabeça linear (*few-shot*)** (a fazer): congela a rede, troca só a última camada (`head`) e treina com
   K = 5, 20 ou todos os exemplos por palavra. Se funcionar, **as features aprendidas em inglês transferem para o português**.
   Treino e teste devem ser separados **por falante**.

In [ ]:
from dados.mswc_pt import PT2EN, listar_mswc_pt, avaliar_cabeca_linear

if TAREFA != "gsc35":
    print("[pulando] §11: o MSWC-pt precisa da tarefa gsc35 (o mapeamento PT -> EN usa dígitos)")
elif RODAR_MSWC and treinados:
    subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "mswc",
                    "--destino", str(PASTAS["dados"]), "--arquivos", str(DOWNLOADS)], check=True)
    palavras, arquivos, rotulos = listar_mswc_pt(PASTAS["dados"] / "mswc" / "pt" / "clips")
    print("palavras usadas:", palavras, "| acaso =", f"{100/len(palavras):.1f}%")
    wavs_pt = [ler_audio(f) for f in arquivos]
    y_pt = torch.tensor(rotulos)
    idx_en = torch.tensor([CLASSES_GSC.index(PT2EN[w]) for w in palavras])

    linhas = []
    for nome, rodadas in treinados.items():
        L = logits_de_wavs(rodadas[0][1], wavs_pt, extrator, MEDIA, DESVIO)
        linha = dict(modelo=nome, zero_shot_mapeado=(L[:, idx_en].argmax(1) == y_pt).float().mean().item())
        try:
            linha.update(avaliar_cabeca_linear(rodadas[0][1], wavs_pt, y_pt))
        except NotImplementedError as e:
            print(f"[a fazer] {e}")
        linhas.append(linha)
    display(pd.DataFrame(linhas).round(4))
else:
    print("[pulando] §11: RODAR_MSWC=False ou nenhum modelo treinado")

**O que observamos:** _(preencher)_

<a name="conclusoes"></a>
## 12. Conclusões (a fazer)

_(Preencher ao final, com os números das seções anteriores.)_

**Hipótese (a): dá para encolher e manter um reconhecimento funcional?**
- [ ] confirmada / refutada. Evidência: §6 (menor modelo acima do critério: ___ com ___ KB e ___ MACs).

**Hipótese (b): o que custa mais, o MFCC ou a rede?**
- [ ] Evidência: §8 (MFCC = ___ ms; rede = ___ a ___ ms; razão MFCC/rede = ___).

**Robustez e generalização:** §9 (ruído) · §10 (outros falantes) · §11 (outro idioma).

**Recomendação para um dispositivo de borda:** _(qual arquitetura e por quê)_

<a name="limitacoes"></a>
## 13. Limitações e próximos passos

- Latência medida na CPU do Colab, não num microcontrolador real (próximo passo: quantização int8 + TFLite Micro).
- Sem *data augmentation* com ruído no treino (o teste DEMAND está fora da distribuição de propósito).
- Subamostras no teste de ruído (3.000 áudios) e no AudioMNIST (100 por falante).
- MSWC-pt: poucas palavras com clipes suficientes; o split precisa ser feito por falante.
- Possíveis extensões: BC-ResNet (Kim et al. 2021), destilação de conhecimento, poda de pesos.

<a name="referencias"></a>
## Referências

- ZHANG, Y. et al. **Hello Edge: Keyword Spotting on Microcontrollers.** arXiv:1711.07128, 2017.
- KIM, B. et al. **Broadcasted Residual Learning for Efficient Keyword Spotting (BC-ResNet).** arXiv:2106.04140, 2021.
- BERG, A. et al. **Keyword Transformer: A Self-Attention Model for Keyword Spotting.** arXiv:2104.00769, 2021.
- TANG, R.; LIN, J. **Deep Residual Learning for Small-Footprint Keyword Spotting.** ICASSP 2018.
- WARDEN, P. **Speech Commands: A Dataset for Limited-Vocabulary Speech Recognition.** arXiv:1804.03209, 2018.
- HE, K. et al. **Deep Residual Learning for Image Recognition.** CVPR 2016.
- VASWANI, A. et al. **Attention Is All You Need.** NeurIPS 2017.
- BENGIO, Y. et al. **A Neural Probabilistic Language Model.** JMLR 2003.
- BETTER STACK. **Speech Recognition Without the Internet Now Fits on a $10 Chip (Whistle).** YouTube, 2026.

Lista completa em [`docs/referencias.md`](https://github.com/gpeuzinCosta/kws-borda/blob/main/docs/referencias.md).